# 02 · Bronze → Silver

Conceitos praticados:
- **Deduplicação** (`dropDuplicates` por chave de negócio)
- **Tratamento de nulos** (nulos estruturais vs. nulos de negócio — nem todo nulo deve ser preenchido!)
- **Conversão de formatos** (string → timestamp, string → decimal/double)
- **Validações de qualidade de dados** com um log de resultados persistido em Delta

In [0]:
catalog = "cine_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

In [0]:
from pyspark.sql import functions as F, Row
from datetime import datetime
from pyspark.sql.window import Window

window = Window.partitionBy("id").orderBy(F.col("ingestion_timestamp").desc())

dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

def converter_data(data_str, formatos_permitidos=None):
    if not formatos_permitidos:
        # Lista dos formatos mais comuns encontrados em bases de dados
        formatos_permitidos = [
            "%d-%m-%Y",
            "%Y-%m-%d",
            "%d/%m/%Y",
            "%Y/%m/%d",
            "%m/%d/%Y",
            "%m-%d-%Y",
        ]

    data_str = data_str.strip()
    for formato in formatos_permitidos:
        try:
            return datetime.strptime(data_str, formato)
        except ValueError:
            pass
    return None

## 1. info

In [0]:
df_bronze_info = spark.table(f"{bronze_schema}.info")

df_silver_info = (
    df_bronze_info
    .withColumn("_rn", F.row_number().over(window))
    .filter((F.col("_rn") == 1) & F.col("id").isNotNull())
    .drop("_rn")
    .withColumn("original_language", F.upper(F.trim(F.col("original_language"))))
    .withColumn("status", F.lower(F.trim(F.col("status"))))
    # .withColumn("release_date", converter_data(F.col("release_date")))

    .withColumn("status",
        F.when(F.col("status") == "released", "Lançado")
        .when(F.col("status") == "post production", "Pós-Produção")
        .when(F.col("status") == "in production", "Em Produção")
        .when(F.col("status") == "planned", "Planejado")
        .when(F.col("status") == "rumored", "Rumores")
        .when(F.col("status") == "canceled", "Cancelado")
        .otherwise("Não Informado")
    )

    .withColumn("release_date",
        F.date_format(
            F.coalesce(
                F.try_to_date(F.col("release_date"), "dd-MM-yyyy"),
                F.try_to_date(F.col("release_date"), "yyyy-MM-dd"),
                F.try_to_date(F.col("release_date"), "dd/MM/yyyy"),
                F.try_to_date(F.col("release_date"), "yyyy/MM/dd"),
                F.try_to_date(F.col("release_date"), "MM/dd/yyyy"),
                F.try_to_date(F.col("release_date"), "MM-dd-yyyy"),
            ),
            "yyyy-MM-dd"
        )
    )

    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("release_date", "data_lancamento")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("status", "status_filme")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")

    .withColumn("ano_lancamento", F.year(F.try_to_date(F.col("data_lancamento"), "yyyy-MM-dd")))
)

dq_check("silver.info", "id_filme não nulo", df_silver_info, F.col("id_filme").isNotNull())
dq_check_unique("silver.info", "id_filme único", df_silver_info, ["id_filme"])

df_silver_info.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.info")

display(df_bronze_info.limit(10))
display(df_silver_info.limit(10))

[PASS] silver.info | id_filme não nulo | 0/97879 linhas falharam
[PASS] silver.info | id_filme único | 0 chaves duplicadas de 97879 linhas


id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_timestamp
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-10-16T22:31:04.718Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-10-16T22:31:04.718Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-10-16T22:31:04.718Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-10-16T22:31:04.718Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-10-16T22:31:04.718Z
284054,tt1825683,Black Panther,Black Panther,en,2018-02-13,135,Released,"King T'Challa returns home to the reclusive, technologically advanced African nation of Wakanda to serve as his country's new leader. However, T'Challa soon finds that he is challenged for the throne by factions within his own country as well as without. Using powers reserved to Wakandan kings, T'Challa assumes the Black Panther mantle to join with ex-girlfriend Nakia, the queen-mother, his princess-kid sister, members of the Dora Milaje (the Wakandan 'special forces') and an American secret agent, to prevent Wakanda from being dragged into a world war.",null,2026-10-16T22:31:04.718Z
284052,tt1211837,Doctor Strange,Doctor Strange,en,2016-10-25,115,Released,"After his career is destroyed, a brilliant but arrogant surgeon gets a new lease on life when a sorcerer takes him under her wing and trains him to defend the world against evil.",The impossibilities are endless.,2026-10-16T22:31:04.718Z
315635,tt2250912,Spider-Man: Homecoming,Spider-Man: Homecoming,en,2017-07-05,133,RELEASED,"Following the events of Captain America: Civil War, Peter Parker, with the help of his mentor Tony Stark, tries to balance his life as an ordinary high school student in Queens, New York City, with fighting crime as his superhero alter ego Spider-Man as a new threat, the Vulture, emerges.",Homework can wait. The city can't.,2026-10-16T22:31:04.718Z
283995,tt3896198,Guardians of the Galaxy Vol. 2,Guardians of the Galaxy Vol. 2,en,2017-04-19,137,Released,The Guardians must fight to keep their newfound family together as 

id_filme,tconst,titulo,titulo_original,idioma_original,data_lancamento,duracao_minutos,status_filme,sinopse,frase_divulgacao,ingestion_timestamp,ano_lancamento
14564,tt0498381,Rings,Rings,EN,2017-02-01,102,Lançado,"\Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her boyfriend and in doing so makes a horrifying discovery: there is a """"\""""movie within the movie""""\"""" that no one has ever seen before.""""""""First you watch it. Then you die.",null,2026-10-16T22:31:04.718Z,2017
32471,tt1587420,Mixtape,Mixtape,EN,2021-12-03,94,Lançado,null,null,2026-10-16T22:31:04.718Z,2021
38258,tt0093119,Grizzly II: Revenge,Grizzly II: Revenge,EN,2020-02-17,74,Lançado,\All hell breaks loose when a giant grizzly,reacting to the slaughter of her cubs by poachers,2026-10-16T22:31:04.718Z,2020
38492,tt0215569,Billy Joel - Live at Yankee Stadium,Billy Joel Live at Yankee Stadium,EN,2022-06-22,86,Lançado,Billy Joel plays his greatest hits in the Big Apple.,null,2026-10-16T22:31:04.718Z,2022
38700,tt1502397,Bad Boys for Life,Bad Boys for Life,EN,2020-01-15,124,Lançado,"Marcus and Mike are forced to confront new threats, career changes, and midlife crises as they join the newly created elite team AMMO of the Miami police department to take down the ruthless Armando Armas, the vicious leader of a Miami drug cartel.",Ride together. Die together.,2026-10-16T22:31:04.718Z,2020
42018,tt0090907,The Horse Thief,盗马贼,ZH,2019-03-19,88,Lançado,"Devout Buddhists, Norbu and Dolma live with their young son Tashi in a clan in Tibet. Norbu is a highwayman. After Norbu is charged with stealing from the temple, he and his family are banished. Impoverished and marginalized, they can do little when their beloved son becomes ill. Tashi dies of a fever. After a second son is born, Norbu focuses his every action on keeping this child alive, seeking re-admission to the clan for his wife and child, then risking all to save them from isolation and starvation in winter.",null,2026-10-16T22:31:04.718Z,2019
42330,tt10443316,Monkey Magic,大闹西游,ZH,2018-09-22,66,Lançado,"Dearth Voyd, the ruler of the dark side of the universe, wants to turn the human world into a place of violence and evil!! However, a huge meteor falls upon Flower-Fruit mountain giving birth to Kongo, a monkey made of stone! Kongo, now sets out on his quest to fight evil and restore order to a desperate land. Episodes 1-3",null,2026-10-16T22:31:04.718Z,2018
43074,tt1289401,Ghostbusters,Ghostbusters,EN,2016-07-14,117,Lançado,"Following a ghost invasion of Manhattan, paranormal enthusiasts Erin Gilbert and Abby Yates, nuclear engineer Jillian Holtzmann, and subway worker Patty Tolan band together to stop the otherworldly threat.",Who You Gonna Call?,2026-10-16T22:31:04.718Z,2016
45033,tt1202339,20 Seconds of Joy,20 Seconds of Joy,DE,2018-01-01,60,Lançado,"Traces the story of an extreme athlete, past and present; but also explores the psychology behind life, death, risk and the confrontation of fear.",null,2026-10-16T22:31:04.718Z,2018
46983,tt0051471,The Song of Styrene,Le Chant du styrène,FR,2022-05-23,13,Lançado,Le chant du Styrène is a 1958 French documentary film directed by Alain Resnais. The film was an order by French industrial group Pechiney to highlight the merits of plastics.,null,2026-10-16T22:31:04.718Z,2022


## 2. financials

In [0]:
df_bronze_financials = spark.table(f"{bronze_schema}.financials")

df_dolar_october_raw = spark.table(f"{bronze_schema}.dolar_october")

w_dolar = Window.partitionBy("cotacao").orderBy(F.col("data_hora").desc())
dolar_atual = (
    df_dolar_october_raw
        .filter(F.col("cotacao").isNotNull())
        .orderBy(F.col("data_hora").desc())
        .collect()[0]["cotacao"]
)

df_silver_financials = (
    df_bronze_financials
    .withColumn("_rn", F.row_number().over(window))
    .filter((F.col("_rn") == 1) & F.col("id").isNotNull())
    .drop("_rn")

    .withColumn("revenue", F.lower(F.trim(F.col("revenue"))))

    .withColumn("budget",
        F.when(F.col("budget") == "0", F.lit(None))
        .when(F.col("budget") == "N/A", F.lit(None))
        .when(F.col("budget").rlike(r"^\d+(\.\d+)?M$"),
              (F.regexp_replace(F.col("budget"), "M$", "").cast("decimal(15,2)") * 1000000))
        .when(F.col("budget").rlike(r"^\d+(\.\d+)?K$"),
              (F.regexp_replace(F.col("budget"), "K$", "").cast("decimal(15,2)") * 1000))
        .when(F.col("budget").rlike(r"^\$\s*\d"),
              F.regexp_replace(F.col("budget"), r"^\$\s*", "").cast("decimal(15,2)"))
        .when(F.col("budget").rlike(r"^USD\s+\d"),
              F.regexp_replace(F.col("budget"), r"^USD\s+", "").cast("decimal(15,2)"))
        .otherwise(F.col("budget").cast("decimal(15,2)"))
    )

    .withColumn("revenue",
        F.when(F.col("revenue") == "não informado", F.lit(None))
        .when(F.col("revenue") == "unknown", F.lit(None))
        .when(F.col("revenue") == "0", F.lit(None))
        .otherwise(F.col("revenue").cast("decimal(15,2)"))
    )

    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("budget", "orcamento_usd")
    .withColumnRenamed("revenue", "receita_usd")

    .withColumnRenamed("orcamento_usd", "orcamento_brl")
    .withColumnRenamed("receita_usd", "receita_brl")

    .withColumn("orcamento_brl", (F.col("orcamento_brl") * dolar_atual).cast("decimal(15,2)"))
    .withColumn("receita_brl", (F.col("receita_brl") * dolar_atual).cast("decimal(15,2)"))

    .withColumn("lucro_brl",
        F.when(F.col("orcamento_brl").isNotNull() & F.col("receita_brl").isNotNull(),
               F.col("receita_brl") - F.col("orcamento_brl"))
        .otherwise(F.lit(None))
    )

    .withColumn(
        "margem_lucro_pct",
        F.when(
            F.col("receita_brl").isNotNull(),
            F.round((F.col("lucro_brl") / F.col("receita_brl")) * 100, 2),
        )
        .otherwise(F.lit(None)),
    )

    .select("id_filme", "orcamento_brl", "receita_brl", "lucro_brl", "margem_lucro_pct")
)

dq_check_unique("silver.financials", "id_filme único", df_silver_financials, ["id_filme"])
dq_check("silver.financials", "id_filme não nulo", df_silver_financials,
         F.col("id_filme").isNotNull())
dq_check("silver.financials", "receita em domínio válido", df_silver_financials,
         F.col("receita_brl").isNotNull())

df_silver_financials.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.financials")

display(df_bronze_financials.limit(10))
display(df_silver_financials.limit(10))

[PASS] silver.financials | id_filme único | 0 chaves duplicadas de 99006 linhas
[PASS] silver.financials | id_filme não nulo | 0/99006 linhas falharam
[FAIL] silver.financials | receita em domínio válido | 95560/99006 linhas falharam


id,budget,revenue,ingestion_timestamp
293660,58000000,Unknown,2026-10-16T22:31:08.163Z
299536,300000000,2052415039,2026-10-16T22:31:08.163Z
299534,356000000,2800000000,2026-10-16T22:31:08.163Z
475557,55000000,1074458282,2026-10-16T22:31:08.163Z
271110,250000000,Não Informado,2026-10-16T22:31:08.163Z
284054,200000000,1349926083,2026-10-16T22:31:08.163Z
284052,180000000,676343174,2026-10-16T22:31:08.163Z
315635,175000000,880166924,2026-10-16T22:31:08.163Z
283995,200000000,863756051,2026-10-16T22:31:08.163Z
297761,175000000,746846894,2026-10-16T22:31:08.163Z


id_filme,orcamento_brl,receita_brl,lucro_brl,margem_lucro_pct
14564,127065000.00,422266931.51,295201931.51,69.91
32471,null,null,null,null
38258,38119500.00,null,null,null
38492,null,null,null,null
38700,457434000.00,2167755553.15,1710321553.15,78.90
42018,null,null,null,null
42330,null,null,null,null
43074,731894400.00,1164665129.24,432770729.24,37.16
45033,1713852.72,null,null,null
46983,null,null,null,null


## 3. metrics

In [0]:
df_bronze_metrics = spark.table(f"{bronze_schema}.metrics")

def to_dec(table, name_col: str):
    return table.withColumn(name_col,
        F.when(F.col(name_col).rlike(r"^\d+([.,]\d+)?$"),
               F.regexp_replace(F.col(name_col), ",", ".").cast("decimal(15,3)"))
        .otherwise(F.lit(None))
    )

def to_int(table, name_col: str):
    return table.withColumn(name_col,
        F.when(F.col(name_col).rlike(r"^\d+$"),
               F.col(name_col).cast("decimal(15,0)"))
        .otherwise(F.lit(None))
    )

df_silver_metrics = (
    df_bronze_metrics
    .withColumn("_rn", F.row_number().over(window))
    .filter((F.col("_rn") == 1) & F.col("id").isNotNull())
    .drop("_rn")
)

df_silver_metrics = to_dec(df_silver_metrics, "popularity")
df_silver_metrics = to_dec(df_silver_metrics, "vote_average")
df_silver_metrics = to_dec(df_silver_metrics, "averageRating")
df_silver_metrics = to_int(df_silver_metrics, "vote_count")
df_silver_metrics = to_int(df_silver_metrics, "numVotes")

df_silver_metrics = (
    df_silver_metrics

    .withColumn("vote_average",
        F.when(F.col("vote_average") < 0, F.lit(None))
        .when(F.col("vote_average") > 10, F.lit(None))
        .otherwise(F.col("vote_average"))
    )

    .withColumn("averageRating",
        F.when(F.col("averageRating") < 0, F.lit(None))
        .when(F.col("averageRating") > 10, F.lit(None))
        .otherwise(F.col("averageRating"))
    )

    .withColumn("vote_count",
        F.when(F.col("vote_count") < 0, F.lit(None))
        .otherwise(F.col("vote_count"))
    )

    .withColumn("numVotes",
        F.when(F.col("numVotes") < 0, F.lit(None))
        .otherwise(F.col("numVotes"))
    )

    .withColumn("popularity",
        F.when(F.col("popularity") < 0, F.lit(None))
        .otherwise(F.col("popularity"))
    )

    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("popularity", "popularidade")
    .withColumnRenamed("vote_average", "nota_media_tmdb")
    .withColumnRenamed("vote_count", "qtd_votos_tmdb")
    .withColumnRenamed("averageRating", "nota_media_imdb")
    .withColumnRenamed("numVotes", "qtd_votos_imdb")
)

dq_check_unique("silver.metrics", "id_filme único", df_silver_financials, ["id_filme"])
dq_check_unique("silver.metrics", "id único", df_silver_metrics,
                 ["id_filme"])

display(df_bronze_metrics.limit(10))
display(df_silver_metrics.limit(10))

[PASS] silver.metrics | id_filme único | 0 chaves duplicadas de 99006 linhas
[PASS] silver.metrics | id único | 0 chaves duplicadas de 99013 linhas


id,popularity,vote_average,vote_count,averageRating,numVotes,ingestion_timestamp
293660,72.735,7.606,28894,8.0,1270339,2026-10-16T22:31:10.663Z
299536,"154,34",8.255,27713,8.4,1406782,2026-10-16T22:31:10.663Z
299534,91.756,8.263,23857,8.4,1484150,2026-10-16T22:31:10.663Z
475557,"54,522",8.168,23425,8.3,1723035,2026-10-16T22:31:10.663Z
271110,70.741,7.4,21541,7.8,947222,2026-10-16T22:31:10.663Z
284054,43.665,7.39,null,7.3,924922,2026-10-16T22:31:10.663Z
284052,70.535,7.427,20935,7.5,895880,2026-10-16T22:31:10.663Z
315635,65.88,7.345,20507,7.4,835116,2026-10-16T22:31:10.663Z
283995,67.553,7.624,20353,7.6,844767,2026-10-16T22:31:10.663Z
297761,35.356,5.909,20097,5.9,null,2026-10-16T22:31:10.663Z


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb,ingestion_timestamp
14564,24.584,4.966,2375,null,46286,2026-10-16T22:31:10.663Z
32471,8.929,7.064,118,6.600,4617,2026-10-16T22:31:10.663Z
38258,null,3.161,28,null,null,2026-10-16T22:31:10.663Z
38492,2.980,7.100,null,7.800,212,2026-10-16T22:31:10.663Z
38700,46.619,7.139,7570,6.500,199420,2026-10-16T22:31:10.663Z
42018,3.403,6.630,27,6.800,1750,2026-10-16T22:31:10.663Z
42330,1.561,10.000,1,6.400,34,2026-10-16T22:31:10.663Z
43074,40.052,5.371,5976,null,260417,2026-10-16T22:31:10.663Z
45033,0.600,8.000,2,null,154,2026-10-16T22:31:10.663Z
46983,0.693,8.500,2,7.000,1209,2026-10-16T22:31:10.663Z


## 4. reviews

In [0]:
df_bronze_reviews = spark.table(f"{bronze_schema}.reviews")

df_silver_reviews = (
    df_bronze_reviews
    .dropDuplicates(["id", "nome", "nota", "comentario"])
    .filter(F.col("id").isNotNull())
    
    .withColumn("nota",
        F.when(F.col("nota") > 10.0, F.lit(None))
        .when(F.col("nota") < 0.0, F.lit(None))
        .otherwise(F.col("nota"))
    )

    .withColumn("comentario",
        F.when(F.col("comentario").isNotNull(), F.col("comentario"))
        .when(F.col("comentario").rlike(r"^\s*$"), F.lit("Sem comentário"))
        .otherwise(F.lit("Sem comentário"))
    )

    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")
    .withColumnRenamed("nota", "nota_usuario")
    .withColumnRenamed("comentario", "comentario_usuario")
)

dq_check_unique("silver.reviews", "review único", df_silver_reviews,
                 ["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])

df_silver_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.reviews")

display(df_bronze_reviews.limit(10))
display(df_silver_reviews.limit(10))


[PASS] silver.reviews | review único | 0 chaves duplicadas de 32412 linhas


id,nome,nota,comentario,ingestion_timestamp
442113,Mariana Cardoso 277,4.4,null,2026-10-16T22:31:16.109Z
637007,Lucas Reis 602,3.9,null,2026-10-16T22:31:16.109Z
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.,2026-10-16T22:31:16.109Z
413036,Gabriela Monteiro 401,7.5,null,2026-10-16T22:31:16.109Z
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.,2026-10-16T22:31:16.109Z
387727,Maria Alves 707,8.2,"Gostei bastante, recomendo.",2026-10-16T22:31:16.109Z
1032506,Amanda Castro 242,4.7,"Fraco, não recomendo.",2026-10-16T22:31:16.109Z
446554,Sandra Rodrigues 736,6.5,Assisti até o final mas não me marcou.,2026-10-16T22:31:16.109Z
569916,Camila Lima 489,8.2,Muito bom! Vale a pena assistir.,2026-10-16T22:31:16.109Z
864552,Roberto Almeida 555,9.3,"Obra-prima do cinema, simplesmente espetacular.",2026-10-16T22:31:16.109Z


id_filme,nome_usuario,nota_usuario,comentario_usuario,ingestion_timestamp
442113,Mariana Cardoso 277,4.4,Sem comentário,2026-10-16T22:31:16.109Z
637007,Lucas Reis 602,3.9,Sem comentário,2026-10-16T22:31:16.109Z
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.,2026-10-16T22:31:16.109Z
413036,Gabriela Monteiro 401,7.5,Sem comentário,2026-10-16T22:31:16.109Z
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.,2026-10-16T22:31:16.109Z
387727,Maria Alves 707,8.2,"Gostei bastante, recomendo.",2026-10-16T22:31:16.109Z
1032506,Amanda Castro 242,4.7,"Fraco, não recomendo.",2026-10-16T22:31:16.109Z
446554,Sandra Rodrigues 736,6.5,Assisti até o final mas não me marcou.,2026-10-16T22:31:16.109Z
569916,Camila Lima 489,8.2,Muito bom! Vale a pena assistir.,2026-10-16T22:31:16.109Z
864552,Roberto Almeida 555,9.3,"Obra-prima do cinema, simplesmente espetacular.",2026-10-16T22:31:16.109Z


## 5. genres

In [0]:
df_bronze_genres = spark.table(f"{bronze_schema}.credits_and_tags")

df_silver_genres = (
    df_bronze_genres
    .withColumn("_rn", F.row_number().over(window))
    .filter((F.col("_rn") == 1) & F.col("id").isNotNull())
    .drop("_rn")
)

dq_check_unique("silver.genres", "id único", df_silver_genres, ["id"])

df_silver_genres = (
    df_bronze_genres
    .withColumn("genres",
        F.when(F.col("genres").rlike(r"\s") & ~F.col("genres").rlike(r"[,;|]"), F.lit(None))
        .when(F.col("genres").endswith(".jpg"), F.lit(None))
        .when(F.col("genres").rlike(r"\d"), F.lit(None))
        .otherwise(F.col("genres"))
    )

    # 1. Substitui os divisores por ", "
    .withColumn("genres",
        F.regexp_replace(F.col("genres"), r"\s*[*;|]\s*", ", "),
    )
    
    # 2. Executa o split
    .withColumn(
        "genres_array",
        F.split(F.trim(F.col("genres")), r"\s*,\s*"),
    )
    
    # 3. Explode o array gerando 1 linha por gênero
    .withColumn("genres", F.explode_outer(F.col("genres_array")))
    
    # 4. Normaliza e limpa o texto do gênero gerado
    .withColumn("genres", F.trim(F.col("genres")))
    
    # Remove strings vazias que possam ter surgido de vírgulas duplas (ex: "Action,,Drama")
    .filter((F.col("genres").isNotNull()) & (F.col("genres") != ""))
    .drop(F.col("genres_array"))
)

dq_check_unique("silver.genres", "id único", df_silver_genres, ["id"])

df_silver_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.genres")

display(df_bronze_genres.limit(10))
display(df_silver_genres.limit(10))

[PASS] silver.genres | id único | 0 chaves duplicadas de 99006 linhas
[FAIL] silver.genres | id único | 40688 chaves duplicadas de 154332 linhas


id,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,cast,ingestion_timestamp
293660,"Action, Adventure, Comedy","20th Century Fox, The Donners' Company, Genre Films",United States of America,English,"superhero, anti hero, mercenary, based on comic, aftercreditsstinger, duringcreditsstinger",Tim Miller,"Rhett Reese, Paul Wernick","Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J. Miller, Gina Carano, Leslie Uggams, Brianna Hildebrand, Stefan Kapičić, Karan Soni, Randal Reeder",2026-10-16T22:31:13.196Z
299536,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, space, battlefield, genocide, magical object, super power, aftercreditsstinger, marvel cinematic universe (mcu), cosmic","Anthony Russo, Joe Russo",N/A,"Robert Downey Jr., Chris Evans, Chris Hemsworth, Josh Brolin, Mark Ruffalo, Scarlett Johansson, Don Cheadle, Benedict Cumberbatch, Tom Holland, Chadwick Boseman",2026-10-16T22:31:13.196Z
299534,"Adventure, Science Fiction, Action",Marvel Studios,United States of America,"English, Japanese, Xhosa","superhero, time travel, space travel, time machine, based on comic, sequel, alien invasion, superhero team, marvel cinematic universe (mcu), alternate timeline, father daughter relationship, sister sister relationship","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Stan Lee, Jack Kirby, Joe Simon, Steve Englehart, Steve Gan, Bill Mantlo, Keith Giffen, Jim Starlin, Larry Lieber, Don Heck","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Josh Brolin, Don Cheadle, Paul Rudd, Benedict Cumberbatch",2026-10-16T22:31:13.196Z
475557,"Crime, Thriller, Drama","Warner Bros. Pictures, Joint Effort, Village Roadshow Pictures, Bron Studios, DC Films","Canada, United States of America",English,"dream, street gang, society, psychopath, clown, villain, based on comic, murder, psychological thriller, criminal mastermind, mental illness, anarchy, character study, clown makeup, subway train, social realism, supervillain, tv host, 1980s, mother son relationship, origin story, falling into madness, depressing",Todd Phillips,"Todd Phillips, Scott Silver, Bob Kane, Bill Finger, Jerry Robinson","Joaquin Phoenix, Robert De Niro, Zazie Beetz, Frances Conroy, Brett Cullen, Shea Whigham, Bill Camp, Glenn Fleshler, Leigh Gill, Josh Pais",2026-10-16T22:31:13.196Z
271110,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"Romanian, English, German, Russian","civil war, superhero, based on comic, sequel, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu), excited","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Joe Simon, Jack Kirby","Chris Evans, Robert Downey Jr., Scarlett Johansson, Sebastian Stan, Anthony Mackie, Don Cheadle, Jeremy Renner, Chadwick Boseman, Paul Bettany, Elizabeth Olsen",2026-10-16T22:31:13.196Z
284054,"Action, Adventure, Science Fiction",Marvel Studios,United States of America,"English, Korean, Swahili, Xhosa","africa, superhero, based on comic, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu)",Ryan Coogler,"Ryan Coogler, Joe Robert Cole, Stan Lee, Jack Kirby","Chadwick Boseman, Michael B. Jordan, Lupita Nyong'o, Danai Gurira, Martin Freeman, Daniel Kaluuya, Letitia Wright, Winston Duke, Sterling K. Brown, Angela Bassett",2026-10-16T22:31:13.196Z
284052,"Action, Adventure, Fantasy",Marvel Studios,UNITED STATES OF AMERICA,English,"magic, superhero, training, time, based on comic, sorcerer, doctor, neurosurgeon, wizard, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu)",Scott Derrickson,N/A,"Benedict Cumberbatch, Chiwetel Ejiofor, Rachel McAdams, Benedict Wong, Mads Mikkelsen, Tilda Swinton, Michael Stuhlbarg, Benjamin Bratt, Scott Adkins, Zara Phythian",2026-10-16T22:31:13.196Z
315635,"Action, Adventure, Science Fiction, Drama","M

id,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,cast,ingestion_timestamp
293660,Action,"20th Century Fox, The Donners' Company, Genre Films",United States of America,English,"superhero, anti hero, mercenary, based on comic, aftercreditsstinger, duringcreditsstinger",Tim Miller,"Rhett Reese, Paul Wernick","Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J. Miller, Gina Carano, Leslie Uggams, Brianna Hildebrand, Stefan Kapičić, Karan Soni, Randal Reeder",2026-10-16T22:31:13.196Z
293660,Adventure,"20th Century Fox, The Donners' Company, Genre Films",United States of America,English,"superhero, anti hero, mercenary, based on comic, aftercreditsstinger, duringcreditsstinger",Tim Miller,"Rhett Reese, Paul Wernick","Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J. Miller, Gina Carano, Leslie Uggams, Brianna Hildebrand, Stefan Kapičić, Karan Soni, Randal Reeder",2026-10-16T22:31:13.196Z
293660,Comedy,"20th Century Fox, The Donners' Company, Genre Films",United States of America,English,"superhero, anti hero, mercenary, based on comic, aftercreditsstinger, duringcreditsstinger",Tim Miller,"Rhett Reese, Paul Wernick","Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J. Miller, Gina Carano, Leslie Uggams, Brianna Hildebrand, Stefan Kapičić, Karan Soni, Randal Reeder",2026-10-16T22:31:13.196Z
299536,Adventure,Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, space, battlefield, genocide, magical object, super power, aftercreditsstinger, marvel cinematic universe (mcu), cosmic","Anthony Russo, Joe Russo",N/A,"Robert Downey Jr., Chris Evans, Chris Hemsworth, Josh Brolin, Mark Ruffalo, Scarlett Johansson, Don Cheadle, Benedict Cumberbatch, Tom Holland, Chadwick Boseman",2026-10-16T22:31:13.196Z
299536,Action,Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, space, battlefield, genocide, magical object, super power, aftercreditsstinger, marvel cinematic universe (mcu), cosmic","Anthony Russo, Joe Russo",N/A,"Robert Downey Jr., Chris Evans, Chris Hemsworth, Josh Brolin, Mark Ruffalo, Scarlett Johansson, Don Cheadle, Benedict Cumberbatch, Tom Holland, Chadwick Boseman",2026-10-16T22:31:13.196Z
299536,Science Fiction,Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, space, battlefield, genocide, magical object, super power, aftercreditsstinger, marvel cinematic universe (mcu), cosmic","Anthony Russo, Joe Russo",N/A,"Robert Downey Jr., Chris Evans, Chris Hemsworth, Josh Brolin, Mark Ruffalo, Scarlett Johansson, Don Cheadle, Benedict Cumberbatch, Tom Holland, Chadwick Boseman",2026-10-16T22:31:13.196Z
299534,Adventure,Marvel Studios,United States of America,"English, Japanese, Xhosa","superhero, time travel, space travel, time machine, based on comic, sequel, alien invasion, superhero team, marvel cinematic universe (mcu), alternate timeline, father daughter relationship, sister sister relationship","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Stan Lee, Jack Kirby, Joe Simon, Steve Englehart, Steve Gan, Bill Mantlo, Keith Giffen, Jim Starlin, Larry Lieber, Don Heck","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Josh Brolin, Don Cheadle, Paul Rudd, Benedict Cumberbatch",2026-10-16T22:31:13.196Z
299534,Science Fiction,Marvel Studios,United States of America,"English, Japanese, Xhosa","superhero, time travel, space travel, time machine, based on comic, sequel, alien invasion, superhero team, marvel cinematic universe (mcu), alternate timeline, father daughter relationship, sister sister relationship","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Stan Lee, Jack Kirby, Joe Simon, Steve Englehart, Steve Gan, Bill Mantlo, Keith Giffen, Jim Starlin, Larry Lieber, Don Heck","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner,

## 6. companies

In [0]:
df_bronze_companies = spark.table(f"{bronze_schema}.credits_and_tags")

def extrair_e_tratar_entidade(df, col_origem, nome_tipo):
    # Trata caso a coluna não exista no DataFrame
    if col_origem not in df.columns:
        return None
        
    return (
        df.filter((F.col(col_origem).isNotNull()) & (F.col(col_origem) != ""))
        # 1. split 
        .withColumn("nome_entidade", F.split(F.col(col_origem), r"\s*[,|;]\s*"))
        # 2. explode
        .withColumn("nome_entidade", F.explode(F.col("nome_entidade")))
        # 3. revisao do texto
        .withColumn("nome_entidade", F.initcap(F.trim(F.col("nome_entidade"))))
        # 4. categoria
        .withColumn("tipo_atuacao", F.lit(nome_tipo))
        # 5. Descarta vazios
        .filter((F.col("nome_entidade") != "") & (F.col("nome_entidade").isNotNull()))
        .select("nome_entidade", "tipo_atuacao")
    )

df_atores = extrair_e_tratar_entidade(df_bronze_companies, "cast", "Ator")
df_roteiristas = extrair_e_tratar_entidade(df_bronze_companies, "writers", "Roteirista")
df_diretores = extrair_e_tratar_entidade(df_bronze_companies, "directors", "Diretor")
df_produtoras = extrair_e_tratar_entidade(df_bronze_companies, "production_companies", "Produtora")

dfs_para_unir = [df for df in [df_atores, df_roteiristas, df_diretores, df_produtoras] if df is not None]

df_silver_companies = dfs_para_unir[0]
for df in dfs_para_unir[1:]:
    df_silver_companies = df_silver_companies.unionByName(df)

df_silver_companies = (
    df_silver_companies
    .dropDuplicates(["nome_entidade", "tipo_atuacao"])
)

dq_check_unique("silver.companies", "papel único", df_silver_companies, ["nome_entidade", "tipo_atuacao"])

df_silver_companies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.companies")

display(df_bronze_companies.limit(10))
display(df_silver_companies.limit(10))

[PASS] silver.companies | papel único | 0 chaves duplicadas de 467762 linhas


id,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,cast,ingestion_timestamp
293660,"Action, Adventure, Comedy","20th Century Fox, The Donners' Company, Genre Films",United States of America,English,"superhero, anti hero, mercenary, based on comic, aftercreditsstinger, duringcreditsstinger",Tim Miller,"Rhett Reese, Paul Wernick","Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J. Miller, Gina Carano, Leslie Uggams, Brianna Hildebrand, Stefan Kapičić, Karan Soni, Randal Reeder",2026-10-16T22:31:13.196Z
299536,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, space, battlefield, genocide, magical object, super power, aftercreditsstinger, marvel cinematic universe (mcu), cosmic","Anthony Russo, Joe Russo",N/A,"Robert Downey Jr., Chris Evans, Chris Hemsworth, Josh Brolin, Mark Ruffalo, Scarlett Johansson, Don Cheadle, Benedict Cumberbatch, Tom Holland, Chadwick Boseman",2026-10-16T22:31:13.196Z
299534,"Adventure, Science Fiction, Action",Marvel Studios,United States of America,"English, Japanese, Xhosa","superhero, time travel, space travel, time machine, based on comic, sequel, alien invasion, superhero team, marvel cinematic universe (mcu), alternate timeline, father daughter relationship, sister sister relationship","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Stan Lee, Jack Kirby, Joe Simon, Steve Englehart, Steve Gan, Bill Mantlo, Keith Giffen, Jim Starlin, Larry Lieber, Don Heck","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Josh Brolin, Don Cheadle, Paul Rudd, Benedict Cumberbatch",2026-10-16T22:31:13.196Z
475557,"Crime, Thriller, Drama","Warner Bros. Pictures, Joint Effort, Village Roadshow Pictures, Bron Studios, DC Films","Canada, United States of America",English,"dream, street gang, society, psychopath, clown, villain, based on comic, murder, psychological thriller, criminal mastermind, mental illness, anarchy, character study, clown makeup, subway train, social realism, supervillain, tv host, 1980s, mother son relationship, origin story, falling into madness, depressing",Todd Phillips,"Todd Phillips, Scott Silver, Bob Kane, Bill Finger, Jerry Robinson","Joaquin Phoenix, Robert De Niro, Zazie Beetz, Frances Conroy, Brett Cullen, Shea Whigham, Bill Camp, Glenn Fleshler, Leigh Gill, Josh Pais",2026-10-16T22:31:13.196Z
271110,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"Romanian, English, German, Russian","civil war, superhero, based on comic, sequel, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu), excited","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Joe Simon, Jack Kirby","Chris Evans, Robert Downey Jr., Scarlett Johansson, Sebastian Stan, Anthony Mackie, Don Cheadle, Jeremy Renner, Chadwick Boseman, Paul Bettany, Elizabeth Olsen",2026-10-16T22:31:13.196Z
284054,"Action, Adventure, Science Fiction",Marvel Studios,United States of America,"English, Korean, Swahili, Xhosa","africa, superhero, based on comic, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu)",Ryan Coogler,"Ryan Coogler, Joe Robert Cole, Stan Lee, Jack Kirby","Chadwick Boseman, Michael B. Jordan, Lupita Nyong'o, Danai Gurira, Martin Freeman, Daniel Kaluuya, Letitia Wright, Winston Duke, Sterling K. Brown, Angela Bassett",2026-10-16T22:31:13.196Z
284052,"Action, Adventure, Fantasy",Marvel Studios,UNITED STATES OF AMERICA,English,"magic, superhero, training, time, based on comic, sorcerer, doctor, neurosurgeon, wizard, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu)",Scott Derrickson,N/A,"Benedict Cumberbatch, Chiwetel Ejiofor, Rachel McAdams, Benedict Wong, Mads Mikkelsen, Tilda Swinton, Michael Stuhlbarg, Benjamin Bratt, Scott Adkins, Zara Phythian",2026-10-16T22:31:13.196Z
315635,"Action, Adventure, Science Fiction, Drama","M

nome_entidade,tipo_atuacao
Sebastian Stan,Ator
Jon Favreau,Ator
Zoë Kravitz,Ator
Frank Schorpion,Ator
Izzie Coffey,Ator
Jack Kesy,Ator
Olivia Cooke,Ator
Cade Woodward,Ator
Jennifer Lawrence,Ator
Dakota Fanning,Ator


## 7. dolar

In [0]:
df_dolar_atual = spark.table(f"{bronze_schema}.dolar_october")

datas_limite = (
    df_dolar_atual
    .select(
        F.min("data_hora").alias("min_data"),
        F.max("data_hora").alias("max_data")
    ).collect()[0]
)

min_data = datas_limite["min_data"]
max_data = datas_limite["max_data"]

df_todos_dias = spark.range(1).select(
    F.explode(
        F.sequence(
            F.to_date(F.lit(min_data)),
            F.to_date(F.lit(max_data)),
            F.expr("INTERVAL '1' DAY")
        )
    ).alias("data_hora")
)

df_silver_dolar = df_todos_dias.join(df_dolar_atual.withColumn("data_hora", F.to_date("data_hora")), on="data_hora", how="left").orderBy("data_hora")

w_dol = (
    Window.orderBy("data_hora")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_silver_dolar = (
    df_silver_dolar
    .withColumn("cotacao",
        F.last("cotacao", ignorenulls=True).over(w_dol)
    )

    .drop(F.col("ingestion_timestamp"))
)

df_silver_dolar.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.dolar_october")

display(df_dolar_atual)
display(df_silver_dolar)

data_hora,cotacao,ingestion_timestamp
2026-10-01T13:10:35.446Z,5.2079,2026-10-16T22:31:18.292Z
2026-10-02T13:03:16.256Z,5.2238,2026-10-16T22:31:18.292Z
2026-10-05T13:07:36.558Z,4.9859,2026-10-16T22:31:18.292Z
2026-10-06T13:03:21.267Z,4.9698,2026-10-16T22:31:18.292Z
2026-10-07T13:05:23.249Z,4.9935,2026-10-16T22:31:18.292Z
2026-10-08T13:08:16.814Z,5.0119,2026-10-16T22:31:18.292Z
2026-10-09T13:07:30.746Z,4.9892,2026-10-16T22:31:18.292Z
2026-10-13T13:04:26.866Z,5.0096,2026-10-16T22:31:18.292Z
2026-10-14T13:05:17.833Z,5.0441,2026-10-16T22:31:18.292Z
2026-10-15T13:05:33.305Z,5.0521,2026-10-16T22:31:18.292Z


data_hora,cotacao
2026-10-01,5.2079
2026-10-02,5.2238
2026-10-03,5.2238
2026-10-04,5.2238
2026-10-05,4.9859
2026-10-06,4.9698
2026-10-07,4.9935
2026-10-08,5.0119
2026-10-09,4.9892
2026-10-10,4.9892
